### RAG Pipelines- Data Ingestion to Vector Database Pipeline 

In [21]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

In [23]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data") 

Found 4 PDF files to process.

Processing file: ..\data\pdf\Attention.pdf
Loaded 44 pages

Processing file: ..\data\pdf\EC4102_MidTerm_Proposal.pdf
Loaded 5 pages

Processing file: ..\data\pdf\Mid_term_applied_econometrics_group_6.pdf
Loaded 5 pages

Processing file: ..\data\pdf\SadakVision — INDIS 2026.pdf
Loaded 16 pages

Total documents loaded: 70


In [24]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax αij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv

### Text Splitting and making Chunks

In [25]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata} ")
    return split_docs
   

In [26]:
chunks = split_documents(all_pdf_documents)

Split 70 documents into 183 chunks

Example chunk:
Content: Q
KV
softmax αij
Attention Mechanisms
in Neural Networks
A Comprehensive Mathematical Treatment
From Theory to Implementation
Hasi Hays
arXiv:2601.03329v1  [cs.LG]  6 Jan 2026...
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': 'Attention.pdf', 'file_type': 'pdf'} 


### Embedding and Vector Store DB

In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
    ):
        """Initialize the EmbeddingManager."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self) -> None:
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully: {self.model_name}. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [29]:
# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1618.28it/s]


Model loaded successfully: all-MiniLM-L6-v2. Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_24112\3387099033.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


### VectorStore

In [30]:
class VectorStore:
    """Handles storage of document embeddings using ChromaDB."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persistent_directory: str = "../data/vector_store"
    ):
        """
        Initialize the VectorStore.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persistent_directory (str): Directory where the vector
                store will be persisted.
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self) -> None:
        """Initialize the persistent ChromaDB vector store."""

        try:
            # Create the directory if it doesn't exist
            os.makedirs(
                self.persistent_directory,
                exist_ok=True
            )

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            # Get existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ) -> None:
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Corresponding embeddings for the documents.
        """

        if not documents or embeddings.size == 0:
            raise ValueError(
                "Documents and embeddings cannot be empty"
            )

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must match"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to the vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique document ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Store original document text
            documents_text.append(
                doc.page_content
            )

            # Convert NumPy array to Python list
            embeddings_list.append(
                embedding.tolist()
            )

        # Add everything to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to the vector store."
            )

            print(
                f"Total documents in collection after addition: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise

In [31]:
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 156


In [32]:
chunks

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax αij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv

In [33]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 183 texts...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches: 100%|██████████| 6/6 [00:08<00:00,  1.33s/it]


Generated embeddings with shape: (183, 384)
Adding 183 documents to the vector store...
Successfully added 183 documents to the vector store.
Total documents in collection after addition: 339


### RAG Retriever from VectorStore (use query to retrieve relevant documents)

In [34]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score required.

        Returns:
            List of dictionaries containing:
            - document ID
            - document content
            - metadata
            - similarity score
            - distance
            - rank
        """

        print(f"Retrieving documents for query: '{query}'")
        print(
            f"Top K: {top_k}, "
            f"Score threshold: {score_threshold}"
        )

        # --------------------------------------------------
        # 1. Generate embedding for the user's query
        # --------------------------------------------------

        query_embedding = (
            self.embedding_manager
            .generate_embeddings([query])[0]
        )

        # --------------------------------------------------
        # 2. Search the vector store
        # --------------------------------------------------

        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            # --------------------------------------------------
            # 3. Process the search results
            # --------------------------------------------------

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                # --------------------------------------------------
                # 4. Process each retrieved document
                # --------------------------------------------------

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # ChromaDB returns cosine distance.
                    # Convert distance to similarity score.
                    similarity_score = 1 - distance

                    # --------------------------------------------------
                    # 5. Apply similarity threshold
                    # --------------------------------------------------

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:

                print("No documents found")

            return retrieved_docs

        except Exception as e:

            print(f"Error during retrieval: {e}")

            return []

In [35]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [36]:
rag_retriever.retrieve("What is sadak vision?")

Retrieving documents for query: 'What is sadak vision?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7b819150_45',
  'content': 'Vision-level accuracy, 89% less compute , plus severity\nSensor only Vision only Fusion (SadakVision)\n0.48\n0.782 0.782 0.823 0.787 0.787\n0.606\n0.785 0.785\nPrecision Recall F1\nSame F1\nas vision-only, by design: the sensor decides what\nthe camera looks at, vision gives the verdict.\n 89%\nframes sent to the detector: 4,620 vs 43,200\nan hour.\n+ Severity\nlow / medium / high from the sensor trace.\nChecked so far on 62 human-labelled events.\nSadakVision · INDIS 2026 10',
  'metadata': {'page': 9,
   'creationdate': '',
   'doc_index': 45,
   'file_type': 'pdf',
   'title': 'SadakVision   INDIS 2026',
   'total_pages': 16,
   'source_file': 'SadakVision — INDIS 2026.pdf',
   'content_length': 464,
   'creator': 'PyPDF',
   'page_label': '10',
   'producer': 'appifact kit',
   'source': '..\\data\\pdf\\SadakVision — INDIS 2026.pdf'},
  'similarity_score': 0.0958300232887268,
  'distance': 0.9041699767112732,
  'rank': 1},
 {'id': 'doc_6abb6

### Intergration VectorDB Context pipeline with LLM output

In [40]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv


# Load environment variables from .env
load_dotenv()


# Get Groq API key
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")


# Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    # model_name="gemma2-9b-it",
    model_name = "openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


# Simple RAG pipeline:
# Retrieve relevant context + generate answer
def rag_simple(query, retriever, llm, top_k=3):

    # 1. Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    # 2. Combine retrieved document chunks into one context
    context = "\n\n".join(
        doc["content"]
        for doc in results
    ) if results else ""

    # 3. Handle case where no relevant documents are found
    if not context:
        print("No relevant documents found for the query.")
        return "No relevant documents found."

    # 4. Create the prompt for the LLM
    prompt = f"""
Use the following context to answer the question concisely.
Only use information from the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    # 5. Send the prompt to the Groq LLM
    response = llm.invoke(prompt)

    # 6. Return the generated answer
    return response.content




In [41]:
# Example usage
query = "What is the Attention Mechanism?"

answer = rag_simple(
    query=query,
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)

print("\nFinal Answer:")
print(answer)

Retrieving documents for query: 'What is the Attention Mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

Final Answer:
**Attention Mechanism**  
A differentiable function that, given a query \(q\), a set of keys \(K=(k_1,\dots ,k_n)\), and values \(V=(v_1,\dots ,v_n)\), computes a weighted sum of the values.  
For each key \(k_i\) a scalar score \(e_i=\text{score}(q,k_i)\) is computed.  
The scores are turned into attention weights  
\[
\alpha_i=\frac{\exp(e_i)}{\sum_{j=1}^n \exp(e_j)},
\]
which satisfy \(\sum_i \alpha_i=1\) and \(\alpha_i\ge0\).  
The output is then  
\[
A(q,K,V)=\sum_{i=1}^n \alpha_i\, v_i .
\]
